# 00 · Attention Lab 导读

**这是一份仓库导读和预测核算，不是 Mac GPU 实验报告。**

目标：从“解释一种机制”走向“在同一模型上检验读取方式的速度、质量与成本”。正式实验由 Python 模块执行；Notebook 展示相同版本的代码、证据与可重放的小例子。

本页不下载模型、不调用 API、不运行 GPU、不创建 GitHub 仓库。

## 1. 找到唯一的项目目录

下面只定位仓库，避免 Notebook 工作目录变化时误读另一份结果。

In [1]:
from pathlib import Path
import json, hashlib, importlib.util, sys, inspect

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / ".attention_lab_bootstrap").is_file()), None)
assert root is not None, "请从 attention-lab 项目目录或 notebooks 子目录打开本页"
print("已找到 Attention Lab。远端是否发布请看实际发布回执，不由目录存在推断。")

已找到 Attention Lab。远端是否发布请看实际发布回执，不由目录存在推断。


## 2. 原始方案、执行决定和实验结果分开读

`docs/archive/PLAN_v3.md` 保留最初方案，`docs/PLAN_CURRENT.md` 是初始执行规格及其修订。后来实际采用的缩减范围、停止决定和交付记录见 `PROJECT_STATUS.md`；不能把方案里列出的全部可能方向当作尚待完成的任务。

下面只核对原始材料的文件指纹。它说明存档没有改变，不证明材料里的全部技术或性能主张都经过本项目验证。

In [2]:
manifest = json.loads((root / "docs/source_manifest.json").read_text(encoding="utf-8"))
for item in manifest["files"]:
    data = (root / item["path"]).read_bytes()
    assert len(data) == item["bytes"]
    assert hashlib.sha256(data).hexdigest() == item["sha256"]
    print(item["path"], "—", item["bytes"], "bytes — hash matched")

docs/archive/PLAN_v3.md — 26663 bytes — hash matched
docs/archive/roofline_original.py — 3445 bytes — hash matched
docs/reference/handbook_with_exercises_updated(4).html — 838364 bytes — hash matched
docs/archive/PLAN_v2.md — 35801 bytes — hash matched
docs/archive/PLAN_training_proposal.md — 27778 bytes — hash matched
docs/archive/attention_review_evidence.zip — 5003 bytes — hash matched
docs/archive/attention_tradeoff_plan_v2.zip — 28520 bytes — hash matched
docs/archive/transformer_teaching_plan.zip — 18748 bytes — hash matched
docs/archive/kv_principle_rerun_tests.txt — 1657 bytes — hash matched


## 3. 调用正式的预测工具，而不是另写一份公式

这里加载 `scripts/roofline.py`。它是已上传原脚本的修订版：ρ=0可用、k截断到候选域、格式标签不固定写BF16、默认不填设备带宽。

下面只看读取预算的实现片段；完整来源仍在同一个Python文件。

In [3]:
source_path = root / "scripts/roofline.py"
spec = importlib.util.spec_from_file_location("attention_lab_roofline_reading", source_path)
roofline = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = roofline
spec.loader.exec_module(roofline)
source_sha = hashlib.sha256(source_path.read_bytes()).hexdigest()
print("预测工具源码 SHA-256:", source_sha)
lines = inspect.getsource(roofline.estimate).splitlines()
start = next(i for i, line in enumerate(lines) if "fixed_count =" in line)
print("\n".join(lines[start:start + 14]))

预测工具源码 SHA-256: 5506a9ec34adfc04bdaef46f4f62f8768378aad87a95afb4705fe56f14da87eb
    fixed_count = min(fixed, n)
    candidate_count = n - fixed_count
    k_requested = round(rho * n)  # Python ties-to-even, matching the supplied script.
    k = min(k_requested, candidate_count)
    selected = fixed_count + k
    per_position = 2 * model.layers * model.kv_heads * model.head_dim * element_bytes
    weights = model.parameters * element_bytes
    dense_kv = per_position * n
    scanned_n = n if scan_scope == "all" else candidate_count
    index_per_position = model.layers * model.kv_heads * r * index_bytes
    index_stored = n * index_per_position
    index_read = scanned_n * index_per_position
    selected_kv = selected * per_position
    d_s1 = index_read + selected_kv


## 4. 先算一遍需要读写多少字节

以原方案中 4B 模型的形状、32K 历史、额外名额比例 1/32、索引宽度 32 做配置算例。本页没有传入带宽，所以不输出耗时预测；也没有加载这个模型。

按位置挑选后读取，可以拆成四项：读索引表、从主缓存取出入选 K/V、把它们写进临时数组、主注意力再读这些数组。**S1** 只数前两项；**S2** 再加后两项。它们是两种读写量核算，不能代替真实计时。

本算例没有模拟缓存命中、选位、投影和运行时调度。对应的本机测量已在第一、二课完成，结果不能由这张字节表直接推出。

In [4]:
prediction = roofline.estimate(roofline.MODELS["Qwen3-4B"], 32768)
for key in ("k", "selected_count", "weights_bytes", "dense_kv_bytes",
            "index_stored_bytes", "d_s1_bytes", "d_s2_bytes",
            "saved_s1_scenario_ms"):
    print(f"{key}: {prediction[key]}")
assert prediction["saved_s1_scenario_ms"] is None
assert prediction["kind"] == "prediction"

k: 1024
selected_count: 1092
weights_bytes: 8044936192
dense_kv_bytes: 4831838208
index_stored_bytes: 603979776
d_s1_bytes: 763748352
d_s2_bytes: 1085792256
saved_s1_scenario_ms: None


## 5. 检查两个边界条件

ρ=0 时只保留开头和近期的固定位置；ρ=1 时最多读取全部已有位置，不能越界。下面直接执行这两项规则检查，不要求读者先作预测，也不把全选当成稀疏提速成绩。

In [5]:
shape = roofline.MODELS["Qwen3-0.6B"]
fixed_only = roofline.estimate(shape, 512, rho=0)
full_selection = roofline.estimate(shape, 512, rho=1)
assert fixed_only["k"] == 0 and fixed_only["selected_count"] == 68
assert full_selection["selected_count"] == 512
assert full_selection["degenerate_full_selection"]
print("ρ=0：", fixed_only["selected_count"], "个位置；ρ=1：", full_selection["selected_count"], "个位置。")

ρ=0： 68 个位置；ρ=1： 512 个位置。


## 6. 已有实验怎样接上这份算例

本页是早期建立的仓库导读和预测工具演示。此后，第一至七课的实验已经完成，正式研究交付已经进入主干；本页没有重新运行它们。

| 接下来阅读 | 能看清什么 |
|---|---|
| [第一课](../reports/01_environment_and_attention.html) | 一份历史位置名单怎样变成 K/V 取行，选位加读取实际要多少时间 |
| [第二课](../reports/02_qwen_integration_and_decode.html) | 这段读取替换到真实模型的哪里，整步还有哪些计算 |
| [第三课](../reports/03_what_is_lost_when_reading_less.html) | 模型收到什么资料和问题，窗口、随机、未训练索引怎样回答 |
| [第四课](../reports/04_can_an_indexer_learn.html) | 固定 Q/K 后，小索引器的一次参数更新怎样改变分数与名单 |
| [第五课](../reports/05_training_all_layer_indexers.html) | 怎样逐层训练各自的索引器，而不更新主体 |
| [第六课](../reports/06_from_teacher_features_to_sparse_states.html) | 放回真实模型后，本层少读怎样改变后层的输入 |
| [第七课](../reports/07_trained_sparse_quality_and_cost.html) | 各方法独立回答新题时的得失，以及实际耗时和缓存成本 |

默认执行只读取既有记录和运行小规模 CPU 检查。第四课另有 12 参数、20 步 CPU 教学更新；不会自动加载 0.6B 模型、重训正式索引器或运行 GPU 基准。